# Manage Jobs — List, Inspect Runs, and Trigger

This notebook provides utilities to:
1. **List all jobs** in the workspace (with pagination)
2. **List runs** for a specific job
3. **Run a job** on demand
4. **Check run status** for a triggered run

Uses the Databricks Jobs API via the Python SDK.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import RunLifeCycleState
import pandas as pd
from datetime import datetime, timezone

# Workspace client (auto-authenticated in notebook context)
w = WorkspaceClient()

def fmt_time(epoch_ms):
    """Convert epoch millis to readable datetime string."""
    if not epoch_ms:
        return ""
    return datetime.fromtimestamp(epoch_ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

In [0]:
# List all jobs (paginated via SDK iterator)
jobs_list = []
for job in w.jobs.list(expand_tasks=False, limit=50):
    jobs_list.append({
        "job_id": job.job_id,
        "name": job.settings.name if job.settings else "",
        "creator": job.creator_user_name or "",
        "created_at": fmt_time(job.created_time),
        "schedule": job.settings.schedule.quartz_cron_expression if job.settings and job.settings.schedule else "Manual",
    })

df_jobs = pd.DataFrame(jobs_list)
print(f"Total jobs found: {len(df_jobs)}")
display(df_jobs)

In [0]:
# Use the first job from the list above
JOB_ID = df_jobs.iloc[0]['job_id'] if not df_jobs.empty else 0

if JOB_ID == 0:
    print("No jobs found in the workspace.")
else:
    runs_list = []
    for run in w.jobs.list_runs(job_id=JOB_ID, limit=20):
        runs_list.append({
            "run_id": run.run_id,
            "run_name": run.run_name or "",
            "state": run.state.life_cycle_state.value if run.state else "",
            "result": run.state.result_state.value if run.state and run.state.result_state else "",
            "start_time": fmt_time(run.start_time),
            "end_time": fmt_time(run.end_time),
            "duration_sec": round((run.end_time - run.start_time) / 1000, 1) if run.end_time and run.start_time else None,
            "run_page_url": run.run_page_url or "",
        })

    df_runs = pd.DataFrame(runs_list)
    print(f"Last {len(df_runs)} runs for job {JOB_ID}:")
    display(df_runs)

In [0]:
# Trigger a job run using the first job from the list
JOB_ID_TO_RUN = int(df_jobs.iloc[0]['job_id']) if not df_jobs.empty else 0

if JOB_ID_TO_RUN == 0:
    print("No jobs found in the workspace.")
else:
    print(f"Triggering job {JOB_ID_TO_RUN}...")
    run_response = w.jobs.run_now(job_id=JOB_ID_TO_RUN)
    print(f"Run triggered successfully!")
    print(f"  Run ID: {run_response.run_id}")
    print(f"  Track at: https://{w.config.host}/#job/{JOB_ID_TO_RUN}/run/{run_response.run_id}")

In [0]:
# Check status of the most recent run for the first job
RUN_ID_TO_CHECK = df_runs.iloc[0]['run_id'] if 'df_runs' in dir() and not df_runs.empty else 0

if RUN_ID_TO_CHECK == 0:
    print("No runs found. Run the 'List Runs' cell first.")
else:
    run = w.jobs.get_run(run_id=RUN_ID_TO_CHECK)
    print(f"Run {run.run_id} for job '{run.run_name}':")
    print(f"  State: {run.state.life_cycle_state.value}")
    if run.state.result_state:
        print(f"  Result: {run.state.result_state.value}")
    if run.state.state_message:
        print(f"  Message: {run.state.state_message}")
    print(f"  Started: {fmt_time(run.start_time)}")
    if run.end_time:
        print(f"  Ended: {fmt_time(run.end_time)}")
        print(f"  Duration: {round((run.end_time - run.start_time) / 1000, 1)}s")
    print(f"  URL: {run.run_page_url}")

In [0]:
# Cancel the most recent run for the first job
RUN_ID_TO_CANCEL = int(df_runs.iloc[0]['run_id']) if 'df_runs' in dir() and not df_runs.empty else 0

if RUN_ID_TO_CANCEL == 0:
    print("No runs found. Run the 'List Runs' cell first.")
else:
    w.jobs.cancel_run(run_id=RUN_ID_TO_CANCEL)
    print(f"Cancel requested for run {RUN_ID_TO_CANCEL}. It may take a moment to terminate.")

In [0]:
# Search for jobs by name (case-insensitive substring match)
SEARCH_TERM = ""  # <-- Enter a search term, e.g. "etl" or "daily"

if not SEARCH_TERM:
    print("Set SEARCH_TERM to filter jobs by name, then re-run this cell.")
else:
    matches = df_jobs[df_jobs['name'].str.contains(SEARCH_TERM, case=False, na=False)]
    print(f"Found {len(matches)} jobs matching '{SEARCH_TERM}':")
    display(matches)